# 04 — Mixed-model association: DRVI factors × CPS

Fits one mixed linear model per (factor, CPS phenotype) pair with a **donor random intercept**.
Two nested models are compared for each factor:

- **pan**: one shared factor slope across all regions (+ region fixed effects)
- **regional**: separate factor slope per region

A likelihood-ratio test between them flags factors whose association with a CPS score
differs across brain regions.

**Units**: donor × brain-region pseudobulk (~530 rows) rather than individual nuclei,
because CPS_Local is constant within a donor × region group.

**Factors**: the 64 DRVI latent dimensions are ReLU-split into 128 half-rectified directions
(factor_{2i−1} = relu(DR_i), factor_{2i} = relu(−DR_i)) so that positive and negative
loadings are tested separately.

**Covariates**: sex and age at donor level.
`library_prep` (902 levels, per-library) does not aggregate to a single value per
donor × region and is excluded.

## 1 — Setup

In [ ]:
from __future__ import annotations

import warnings

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
import statsmodels.formula.api as smf
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

import sst_drvi as S

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_rows", 140)

In [ ]:
CANDIDATES = ["sst_k64_covar_long", "sst_k64_covar", "sst_k64"]
MODEL = "drvi"

RUN = None
for candidate in CANDIDATES:
    embed_path = S.EMBED_DIR / f"{candidate}_{MODEL}_embed.h5ad"
    if embed_path.exists():
        RUN = candidate
        break

assert RUN is not None, (
    "No embedding found; searched:\n"
    + "\n".join(str(S.EMBED_DIR / f"{c}_{MODEL}_embed.h5ad") for c in CANDIDATES)
)
print(f"Using run: {RUN}")
embed = ad.read_h5ad(embed_path)
print(embed)

## 2 — Build donor × region pseudobulk

In [ ]:
# Extract latent matrix (n_cells × n_latent).
X = embed.X.toarray() if sp.issparse(embed.X) else np.asarray(embed.X)
n_dims = X.shape[1]
print(f"Latent dimensions: {n_dims}")

# Carry the columns we need; rename for statsmodels formula compatibility (no spaces).
obs = embed.obs[
    ["Donor ID", "Brain Region", "Sex", "Age at Death", *S.CPS_COLUMNS]
].copy()
obs = obs.rename(columns={
    "Donor ID": "donor",
    "Brain Region": "region",
    "Sex": "sex",
    "Age at Death": "age",
})

# ReLU split: factor_{2i-1} = relu(DR_i), factor_{2i} = relu(-DR_i),  i = 1 .. n_dims.
for i in range(n_dims):
    obs[f"factor_{2 * i + 1}"] = np.maximum(X[:, i], 0.0)
    obs[f"factor_{2 * i + 2}"] = np.maximum(-X[:, i], 0.0)

factor_columns = [f"factor_{j}" for j in range(1, 2 * n_dims + 1)]

# Aggregate to donor × region.
# Factors → mean activation per unit; CPS and covariates → first (constant within group).
agg_keys = {f: "mean" for f in factor_columns}
agg_keys |= {c: "first" for c in [*S.CPS_COLUMNS, "sex", "age"]}

dat = obs.groupby(["donor", "region"]).agg(agg_keys).reset_index()

print(
    f"Pseudobulk: {len(dat):,} donor×region units, "
    f"{dat['donor'].nunique()} donors, "
    f"{dat['region'].nunique()} regions"
)
dat.head()

## 3 — Mixed-model helper

Within each region the factor is z-scored before fitting (one-SD change has the same meaning
in every region).  The pan model includes region fixed effects; the regional model replaces the
single slope with region-specific slopes.  Their difference in log-likelihood is tested as χ².

In [ ]:
def test_factor(
    data: pd.DataFrame,
    factor: str,
    phenotype: str,
    covariates: list[str],
) -> tuple[dict, pd.DataFrame]:
    df = data[
        ["donor", "region", phenotype, factor, *covariates]
    ].dropna().copy()

    # A one-SD change has the same meaning in every region.
    df["factor_z"] = df.groupby("region")[factor].transform(
        lambda x: (x - x.mean()) / x.std(ddof=0)
    )

    covariate_formula = (
        " + " + " + ".join(covariates) if covariates else ""
    )

    # One shared factor slope across all regions.
    pan_formula = (
        f"{phenotype} ~ factor_z + C(region)"
        f"{covariate_formula}"
    )
    pan_fit = smf.mixedlm(
        pan_formula,
        data=df,
        groups=df["donor"],
    ).fit(reml=False, method="lbfgs")

    # Separate factor slope for every region.
    regional_formula = (
        f"{phenotype} ~ 0 + C(region) "
        f"+ C(region):factor_z"
        f"{covariate_formula}"
    )
    regional_fit = smf.mixedlm(
        regional_formula,
        data=df,
        groups=df["donor"],
    ).fit(reml=False, method="lbfgs")

    # Likelihood-ratio test: do factor slopes differ across regions?
    lr_stat = max(0.0, 2 * (regional_fit.llf - pan_fit.llf))
    lr_df = len(regional_fit.fe_params) - len(pan_fit.fe_params)
    heterogeneity_p = chi2.sf(lr_stat, lr_df)

    overall = {
        "factor": factor,
        "n_observations": len(df),
        "n_donors": df["donor"].nunique(),
        "pan_beta": pan_fit.fe_params["factor_z"],
        "pan_se": pan_fit.bse_fe["factor_z"],
        "pan_p": pan_fit.pvalues["factor_z"],
        "heterogeneity_lr": lr_stat,
        "heterogeneity_df": lr_df,
        "heterogeneity_p": heterogeneity_p,
        "pan_converged": pan_fit.converged,
        "regional_converged": regional_fit.converged,
    }

    rows = []
    for term in regional_fit.fe_params.index:
        if term.endswith(":factor_z"):
            # Example term: C(region)[Heart]:factor_z
            region = term.split("[", maxsplit=1)[1].split("]", maxsplit=1)[0]

            beta = regional_fit.fe_params[term]
            se = regional_fit.bse_fe[term]

            rows.append(
                {
                    "factor": factor,
                    "region": region,
                    "beta": beta,
                    "se": se,
                    "ci_low": beta - 1.96 * se,
                    "ci_high": beta + 1.96 * se,
                    "p": regional_fit.pvalues[term],
                }
            )

    return overall, pd.DataFrame(rows)

## 4 — Run across all factors and CPS phenotypes

Multiple-testing correction is applied per phenotype × testing family:
- `pan_fdr`: Benjamini–Hochberg over all 128 pan p-values
- `heterogeneity_fdr`: BH over all 128 LRT p-values
- `regional_fdr`: BH over all factor × region p-values

In [ ]:
# Three local CPS phenotypes (see 03_progression.ipynb).
LOCAL_SCORES = {
    "CPS_Local": "composite",
    "CPS_Local_ABeta": "ABeta",
    "CPS_Local_pTau": "pTau",
}

# Sex and age are available at donor level; library_prep excluded (per-library).
covariates = ["age", "C(sex)"]

all_overall, all_regional = [], []

for phenotype, label in LOCAL_SCORES.items():
    print(f"\n=== {phenotype} ({label}) — {len(factor_columns)} factors ===")

    ph_overall, ph_regional = [], []
    for factor in factor_columns:
        overall, regional = test_factor(
            data=dat,
            factor=factor,
            phenotype=phenotype,
            covariates=covariates,
        )
        overall["phenotype"] = phenotype
        regional["phenotype"] = phenotype
        ph_overall.append(overall)
        ph_regional.append(regional)

    # Correct the three families separately within this phenotype.
    ph_df = pd.DataFrame(ph_overall)
    ph_df["pan_fdr"] = multipletests(ph_df["pan_p"], method="fdr_bh")[1]
    ph_df["heterogeneity_fdr"] = multipletests(ph_df["heterogeneity_p"], method="fdr_bh")[1]

    ph_reg = pd.concat(ph_regional, ignore_index=True)
    ph_reg["regional_fdr"] = multipletests(ph_reg["p"], method="fdr_bh")[1]

    n_pan = (ph_df["pan_fdr"] < 0.05).sum()
    n_het = (ph_df["heterogeneity_fdr"] < 0.05).sum()
    n_reg = (ph_reg["regional_fdr"] < 0.05).sum()
    print(
        f"  pan FDR<0.05: {n_pan}  |  "
        f"heterogeneous FDR<0.05: {n_het}  |  "
        f"regional FDR<0.05: {n_reg}"
    )

    all_overall.append(ph_df)
    all_regional.append(ph_reg)

overall_results = pd.concat(all_overall, ignore_index=True)
regional_results = pd.concat(all_regional, ignore_index=True)

## 5 — Results

Interpreting the tests:
- `pan_fdr < 0.05`, heterogeneity not significant → shared pan-region association.
- `heterogeneity_fdr < 0.05` → factor effects differ among regions.
- `regional_fdr < 0.05` → evidence for an association in that particular region.
- Both pan and heterogeneity significant → broadly associated, but effect size varies by region.

Because `statsmodels.MixedLM` uses asymptotic tests, validate important hits with a
donor-level bootstrap or permutation, especially if there is substantial missingness or
poor model convergence.

In [ ]:
# Convergence summary.
n_pan_fail = (~overall_results["pan_converged"]).sum()
n_reg_fail = (~overall_results["regional_converged"]).sum()
print(f"Pan model non-converged: {n_pan_fail} / {len(overall_results)}")
print(f"Regional model non-converged: {n_reg_fail} / {len(overall_results)}")

In [ ]:
# Pan-region hits: shared slope, FDR<0.05, heterogeneity not significant.
pan_hits = overall_results[
    (overall_results["pan_fdr"] < 0.05)
    & ~(overall_results["heterogeneity_fdr"] < 0.05)
].sort_values(["phenotype", "pan_fdr"])

print(f"Pan-region hits: {len(pan_hits)}")
pan_hits[
    ["phenotype", "factor", "pan_beta", "pan_se", "pan_p", "pan_fdr", "n_donors", "pan_converged"]
]

In [ ]:
# Heterogeneous hits: effect size varies by region.
het_hits = overall_results[
    overall_results["heterogeneity_fdr"] < 0.05
].sort_values(["phenotype", "heterogeneity_fdr"])

print(f"Heterogeneous hits: {len(het_hits)}")
het_hits[
    ["phenotype", "factor", "pan_beta", "pan_fdr", "heterogeneity_lr", "heterogeneity_fdr", "regional_converged"]
]

In [ ]:
# Region-specific hits.
reg_hits = regional_results[
    regional_results["regional_fdr"] < 0.05
].sort_values(["phenotype", "regional_fdr"])

print(f"Region-specific hits: {len(reg_hits)}")
reg_hits[
    ["phenotype", "factor", "region", "beta", "se", "ci_low", "ci_high", "p", "regional_fdr"]
]